<a href="https://colab.research.google.com/github/realshubhamraut/CDAC-DBDA-coursework/blob/main/08.advanced-analytics-stats/assignments/case_study_customer_churn/case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Data Preparation

In [ ]:
%pip install colorama                # for style in output
%pip install tabulate
from colorama import Fore, Style
from tabulate import tabulate        # for tabular response view
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import chi2_contingency, f, norm, t
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats import weightstats as ssw
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')
import os

installs required packages and imports libraries for data manipulation, visualization, and statistical analysis

In [ ]:
# datasets stored on github lfs storage and fetching directly if not available on runtime on colab

folder_path = "/content/datasets"
os.makedirs(folder_path, exist_ok=True)

base_url = "https://media.githubusercontent.com/media/realshubhamraut/CDAC-DBDA-coursework/refs/heads/main/08.advanced-analytics-stats/data/"

files = [
    "credit_risk_applicants.csv",
    "credit_risk_metadata.csv",
    "credit_risk_previous_loans.csv"]

for f in files:
    url = base_url + f
    save_path = os.path.join(folder_path, f)
    if not os.path.exists(save_path):
        print(f"downloading: {f}")
        !wget -q "{url}" -O "{save_path}"

print("download complete")


downloads credit risk datasets from github lfs storage into the colab runtime environment

In [ ]:
app = pd.read_csv(r'datasets/credit_risk_applicants.csv')
loans = pd.read_csv(r'datasets/credit_risk_previous_loans.csv')

In [ ]:
meta = pd.read_csv(r'datasets/credit_risk_metadata.csv', encoding='latin-1')

In [ ]:
print(tabulate(meta, headers="keys", tablefmt="grid",showindex=False))


In [ ]:
app.shape, loans.shape

In [ ]:
pd.options.display.max_columns = None

In [ ]:
app.head()

In [ ]:
app.describe()

## 2. Working with missing values

In [ ]:
def missing_overview(df1, name1, df2, name2):
    p1 = (df1.isna().sum() / len(df1) * 100).round(2)
    p2 = (df2.isna().sum() / len(df2) * 100).round(2)

    cats = [
        (">40% missing",    (p1 > 40).sum(),                 (p2 > 40).sum()),
        ("10–40% missing",  ((p1 > 10) & (p1 <= 40)).sum(),  ((p2 > 10) & (p2 <= 40)).sum()),
        ("0–10% missing",   ((p1 > 0) & (p1 <= 10)).sum(),   ((p2 > 0) & (p2 <= 10)).sum()),
        ("0% missing",      (p1 == 0).sum(),                 (p2 == 0).sum())
    ]

    rows = [
        [cat,
         f"{c1} ({c1/len(df1.columns)*100:.2f}%)",
         f"{c2} ({c2/len(df2.columns)*100:.2f}%)"]
        for cat, c1, c2 in cats
    ]

    print(Fore.BLUE + f"\nmissing overview: {name1} vs {name2}\n" + Style.RESET_ALL)
    print(tabulate(rows,
                   headers=["missing values", name1, name2],
                   tablefmt="grid"))


defines a helper function to create a comparative overview of missing values across two datasets

In [ ]:
missing_overview(app, "applicants", loans, "previous loans")

displays a summary table showing missing value distribution categories for both datasets

In [ ]:
app_reasons = {
    "COMMONAREA_AVG": "shared building area; strong proxy for living quality and socio economic status",
    "FONDKAPREMONT_MODE": "building maintenance fund type; indicates infrastructure condition",
    "YEARS_BUILD_AVG": "construction year; powerful indicator of building age and quality",
    "OWN_CAR_AGE": "car age reflects financial stability and lifestyle; missing often means no car",
    "LANDAREA_AVG": "land area size; strong wealth and property indicator",
    "BASEMENTAREA_AVG": "basement size reveals property structure and affluence",
    "EXT_SOURCE_1": "external risk score; known to be one of the most predictive features",
    "NONLIVINGAREA_AVG": "non living area; represents additional property space and wealth",
    "ELEVATORS_AVG": "number of elevators; strong signal of building modernity and urban living",
    "WALLSMATERIAL_MODE": "building wall type; correlates with construction quality and risk",
    "APARTMENTS_AVG": "number of apartments; building density and urban housing signal",
    "ENTRANCES_AVG": "entrances count; reflects building size and occupancy pattern",
    "LIVINGAREA_AVG": "main living area size; strong proxy for household financial status",
    "HOUSETYPE_MODE": "type of house; key segmentation variable for socio economic groups",
    "FLOORSMAX_AVG": "building height; indicator of development, infrastructure, and environment",
    "YEARS_BEGINEXPLUATATION_AVG": "building operational age; structural and environmental risk proxy",
    "TOTALAREA_MODE": "aggregated property size indicator; compact strong feature",
    "EMERGENCYSTATE_MODE": "emergency condition flag; rare but high risk signal when present",
}

prev_reasons = {
    "AMT_DOWN_PAYMENT": "down payment amount; key affordability and prior-loan structure signal",
    "RATE_DOWN_PAYMENT": "interest rate on down payment; reflects credit terms and risk category",
    "NAME_TYPE_SUITE": "context of application (alone, spouse, group); behavioural risk indicator",
    "DAYS_FIRST_DRAWING": "first drawing date; part of loan lifecycle and behavioural timing",
    "DAYS_FIRST_DUE": "first due date; repayment schedule signal",
    "DAYS_LAST_DUE_1ST_VERSION": "original last due date; intended loan duration marker",
    "DAYS_LAST_DUE": "actual last due date; real behavioural repayment length",
    "DAYS_TERMINATION": "termination date; helps identify early closures or irregularities",
    "NFLAG_INSURED_ON_APPROVAL": "whether previous loan was insured; major risk-management attribute",
}

defines dictionaries containing business justifications for keeping high-missing columns that are predictively important

In [ ]:
reason_lookup = {**app_reasons, **prev_reasons}

# lists of kept columns
keep_applicants = set(app_reasons.keys())
keep_previous = set(prev_reasons.keys())


def missing_table(df1, name1, df2, name2):

    p1 = (df1.isna().sum() / len(df1) * 100).round(2)
    p2 = (df2.isna().sum() / len(df2) * 100).round(2)

    cols = sorted(set(df1.columns) | set(df2.columns))

    rows = []

    for col in cols:

        val1 = f"{p1[col]:.2f}%" if col in df1.columns else "-"
        val2 = f"{p2[col]:.2f}%" if col in df2.columns else "-"

        # Initialize flag_colored and reason with default values
        flag_colored = Fore.RED + "drop" + Style.RESET_ALL # Default to 'drop'
        reason = "" # Default empty reason

        # Check if the column should be kept and update values if so
        if col in keep_applicants or col in keep_previous:
            flag_colored = Fore.BLUE + "keep" + Style.RESET_ALL
            reason = reason_lookup.get(col, "") # Retrieve reason, default to empty string if not found

            # Only append the row if the flag is 'keep'
            rows.append([col, val1, val2, flag_colored, reason])

    headers = [
        Fore.BLUE + "column" + Style.RESET_ALL,
        Fore.BLUE + f"{name1} missing %" + Style.RESET_ALL,
        Fore.BLUE + f"{name2} missing %" + Style.RESET_ALL,
        Fore.BLUE + "flag" + Style.RESET_ALL,
        Fore.BLUE + "reason" + Style.RESET_ALL,
    ]

    print()
    print(Fore.BLUE + f"full missingness table: {name1} vs {name2}" + Style.RESET_ALL)
    print()

    print(tabulate(
        rows,
        headers=headers,
        tablefmt="grid",
        showindex=False
    ))

defines a function to generate a detailed missing values table with keep or drop flags and business reasons

In [ ]:
missing_table(app, "applicants", loans, "previous loans")


displays the comprehensive missing values table with flags and business justifications for all columns

In [ ]:
# copies
app_imp   = app.copy()
loans_imp = loans.copy()

# columns to keep, can be used for predictions
app_num_keep = ["COMMONAREA_AVG","YEARS_BUILD_AVG","OWN_CAR_AGE","LANDAREA_AVG","BASEMENTAREA_AVG", "EXT_SOURCE_1","NONLIVINGAREA_AVG","ELEVATORS_AVG","APARTMENTS_AVG","ENTRANCES_AVG", "LIVINGAREA_AVG","FLOORSMAX_AVG","YEARS_BEGINEXPLUATATION_AVG","TOTALAREA_MODE"]
app_cat_keep = ["FONDKAPREMONT_MODE","WALLSMATERIAL_MODE","HOUSETYPE_MODE","EMERGENCYSTATE_MODE",]

loans_num_keep = ["AMT_DOWN_PAYMENT","RATE_DOWN_PAYMENT","DAYS_FIRST_DRAWING","DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION","DAYS_LAST_DUE","DAYS_TERMINATION"]
loans_cat_keep = ["NAME_TYPE_SUITE","NFLAG_INSURED_ON_APPROVAL"]

# keep only existing columns
app_num_keep   = [c for c in app_num_keep   if c in app_imp.columns]
app_cat_keep   = [c for c in app_cat_keep   if c in app_imp.columns]
loans_num_keep = [c for c in loans_num_keep if c in loans_imp.columns]
loans_cat_keep = [c for c in loans_cat_keep if c in loans_imp.columns]

# numeric → median
for df, cols in [(app_imp, app_num_keep), (loans_imp, loans_num_keep)]:
    if cols:
        df[cols] = df[cols].fillna(df[cols].median())

# categorical → mode
def fill_mode(df, cols):
    for col in cols:
       if df[col].dtype == "object":
          if df[col].isna().any():
              df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

fill_mode(app_imp, app_cat_keep)
fill_mode(loans_imp, loans_cat_keep)


imputes missing values using median for numeric columns and mode for categorical columns in the kept features

#### feature engineering note

the following transformations were applied to improve interpretability and model usefulness:

- **age_years**: derived by converting `days_birth` (stored as negative days) into positive age in years.
- **employment_years**: converted `days_employed` into years and removed unrealistic placeholder values (employment > 100 years).
- **credit_income_ratio**: measures how large the requested credit amount is relative to the applicant’s income.
- **annuity_income_ratio**: captures payment burden by comparing monthly annuity to total income.
|

In [ ]:
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)
app = app[app['CODE_GENDER'] != 'XNA']


app['AGE_YEARS']            = -app['DAYS_BIRTH'] / 365
app['EMPLOYMENT_YEARS']     = (-app['DAYS_EMPLOYED'] / 365).mask(lambda x: x > 100, np.nan)
app['EMPLOYMENT_YEARS']     = pd.to_numeric(app['EMPLOYMENT_YEARS'], errors='coerce')            # for fixing boxplot
app['CREDIT_INCOME_RATIO']  = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']

# app.drop(columns=[c for c in ['DAYS_BIRTH','DAYS_EMPLOYED','AMT_CREDIT','AMT_INCOME_TOTAL','AMT_ANNUITY'] if c in app.columns], inplace=True)


engineers new features including age, employment years, and financial ratios for better interpretability

In [ ]:
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series.clip(lower, upper)

app['AMT_INCOME_TOTAL_CLIPPED'] = cap_outliers(app['AMT_INCOME_TOTAL'])

defines a function to cap outliers using the iqr method and applies it to income data

In [ ]:
app.head()

displays the first few rows to verify the new engineered features and outlier-capped columns

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
app['TARGET'].value_counts().plot(kind='bar', ax=ax[0], color=['#66c2a5', '#fc8d62'])
ax[0].set_title('target distribution')
ax[0].set_xlabel('target')
ax[0].set_ylabel('count')
ax[0].set_xticklabels(['non-defaulter', 'defaulter'], rotation=0)

app['TARGET'].value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'])
ax[1].set_title('default rate')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

visualizes the target variable distribution using bar and pie charts to show class imbalance

#### univariate analysis - numerical features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cols = ["AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AGE_YEARS","EMPLOYMENT_YEARS"]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
ax = ax.ravel()

for i, c in enumerate(cols):
    d = app[c].dropna()
    d = d[d <= np.percentile(d, 99)]
    sns.histplot(d, bins=50, kde = True, ax=ax[i], color="steelblue")
    ax[i].set_title(c.lower())

ax[-1].axis("off")
plt.tight_layout()
plt.show()


creates histograms with kde for key numerical features after removing extreme outliers at 99th percentile

#### univariate analysis - categorical features


In [ ]:
cat_cols = ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    app[col].value_counts().plot(kind='barh', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('count')

plt.tight_layout()
plt.show()

creates horizontal bar charts showing the distribution of key categorical features

#### bivariate analysis - income vs default

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# boxplot
app.boxplot(
    column='AMT_INCOME_TOTAL_CLIPPED',
    by='TARGET',
    ax=axes[0]
)
axes[0].set_title('income distribution by default status (clipped)')
axes[0].set_xlabel('target (0=non-defaulter / 1=defaulter)')
axes[0].set_ylabel('income')
axes[0].ticklabel_format(axis='y', style='plain')

# histogram overlay
for t, c in zip([0, 1], ['#4dabf7', '#ff6b6b']):
    subset = app[app['TARGET'] == t]['AMT_INCOME_TOTAL_CLIPPED']
    axes[1].hist(subset, bins=40, alpha=0.6, label=f'target={t}', color=c)

axes[1].set_title('income distribution overlay (clipped)')
axes[1].set_xlabel('income')
axes[1].set_ylabel('frequency')
axes[1].legend()

plt.suptitle('')
plt.tight_layout()
plt.show()


compares income distribution between defaulters and non-defaulters using boxplot and overlayed histograms

#### bivariate analysis - credit amount vs default


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

app.boxplot(column='AMT_CREDIT', by='TARGET', ax=axes[0])
axes[0].set_title('credit amount by default status')
axes[0].set_xlabel('target')
axes[0].set_ylabel('credit amount')
axes[0].set_xticklabels(['non-defaulter', 'defaulter'])
plt.suptitle('')

app.boxplot(column='CREDIT_INCOME_RATIO', by='TARGET', ax=axes[1])
axes[1].set_title('credit-to-income ratio by default status')
axes[1].set_xlabel('target')
axes[1].set_ylabel('ratio')
axes[1].set_xticklabels(['non-defaulter', 'defaulter'])
plt.suptitle('')

plt.tight_layout()
plt.show()

visualizes credit amount and credit-to-income ratio differences between defaulters and non-defaulters

#### bivariate analysis - age vs default

In [ ]:
app['AGE_GROUP'] = pd.cut(app['AGE_YEARS'], bins=[0, 25, 35, 45, 55, 100], labels=['<25', '25-35', '35-45', '45-55', '55+'])
age_default = pd.crosstab(app['AGE_GROUP'], app['TARGET'], normalize='index') * 100

age_default.plot(kind='bar', stacked=False, figsize=(10, 5))
plt.title('default rate by age group')
plt.xlabel('age group')
plt.ylabel('percentage')
plt.legend(['non-defaulter', 'defaulter'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

creates age groups and shows default rate variation across different age categories

#### bivariate analysis - education vs default


In [ ]:
edu_default = pd.crosstab(app['NAME_EDUCATION_TYPE'], app['TARGET'], normalize='index') * 100

edu_default.plot(kind='barh', stacked=False, figsize=(10, 5))
plt.title('default rate by education level')
plt.xlabel('percentage')
plt.ylabel('education type')
plt.legend(['non-defaulter', 'defaulter'])
plt.tight_layout()
plt.show()

displays default rates across different education levels using a horizontal bar chart

#### bivariate analysis - income type vs default

In [ ]:
income_default = pd.crosstab(app['NAME_INCOME_TYPE'], app['TARGET'], normalize='index') * 100

income_default.plot(kind='barh', stacked=False, figsize=(10, 5))
plt.title('default rate by income type')
plt.xlabel('percentage')
plt.ylabel('income type')
plt.legend(['non-defaulter', 'defaulter'])
plt.tight_layout()
plt.show()

shows how default rates vary across different income types such as working, pensioner, and business

#### bivariate analysis - gender vs default

In [ ]:
gender_default = pd.crosstab(app['CODE_GENDER'], app['TARGET'], normalize='index') * 100

gender_default.plot(kind='bar', stacked=False, figsize=(8, 5))
plt.title('default rate by gender')
plt.xlabel('gender')
plt.ylabel('percentage')
plt.legend(['non-defaulter', 'defaulter'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

compares default rates between male and female applicants using a bar chart

#### bivariate analysis - external sources vs default

In [ ]:
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(ext_sources):
    app.boxplot(column=col, by='TARGET', ax=axes[i])
    axes[i].set_title(f'{col} by default status')
    axes[i].set_xlabel('target')
    axes[i].set_ylabel('score')
    axes[i].set_xticklabels(['non-defaulter', 'defaulter'])

plt.suptitle('')
plt.tight_layout()
plt.show()

visualizes the three external credit bureau scores by default status using boxplots

#### correlation analysis

In [ ]:
corr_cols = ['TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AGE_YEARS', 'EMPLOYMENT_YEARS', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'CREDIT_INCOME_RATIO']
corr_matrix = app[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=1)
plt.title('correlation matrix')
plt.tight_layout()
plt.show()

creates a correlation heatmap showing relationships between numerical features and the target variable

In [ ]:
corr_matrix['TARGET'].sort_values(ascending=False)

displays correlations with target sorted in descending order to identify strongest predictors

#### multivariate analysis - income vs credit vs default

In [ ]:
df = app.copy()
i99 = np.percentile(df['AMT_INCOME_TOTAL'], 99.9)
c99 = np.percentile(df['AMT_CREDIT'], 99.9)

df = df[(df['AMT_INCOME_TOTAL'] <= i99) & (df['AMT_CREDIT'] <= c99)]

plt.figure(figsize=(10, 6))

for target in [0, 1]:
    subset = df[df['TARGET'] == target]
    plt.scatter(
        subset['AMT_INCOME_TOTAL'],
        subset['AMT_CREDIT'],
        alpha=0.3,
        label=f"target={target}"
    )

plt.title("income vs credit amount by default status (trimmed outliers)")
plt.xlabel("income")
plt.ylabel("credit amount")
plt.legend()
plt.tight_layout()
plt.show()


creates a scatter plot of income vs credit amount colored by default status after trimming extreme outliers

#### previous loans analysis

In [ ]:
loans['NAME_CONTRACT_STATUS'].value_counts()

displays the distribution of previous loan contract statuses such as approved, refused, and cancelled

In [ ]:
loan_status_counts = loans.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].value_counts().unstack(fill_value=0)
app_merged = app.merge(loan_status_counts, left_on='SK_ID_CURR', right_index=True, how='left')

# Identify the columns that were added from loan_status_counts
new_status_cols = loan_status_counts.columns.tolist()

# Fill NaNs only in these new columns with 0 and convert them to integer type
app_merged[new_status_cols] = app_merged[new_status_cols].fillna(0).astype(int)


aggregates previous loan status counts per customer and merges with applicant data for analysis

In [ ]:
if 'Refused' in app_merged.columns:
    app_merged.boxplot(column='Refused', by='TARGET', figsize=(8, 5))
    plt.title('previous refusals by default status')
    plt.suptitle('')
    plt.xlabel('target')
    plt.ylabel('number of refusals')
    plt.xticks([1, 2], ['non-defaulter', 'defaulter'])
    plt.tight_layout()
    plt.show()

visualizes the number of previous loan refusals between defaulters and non-defaulters using a boxplot

#### hypothesis testing setup

#### hypothesis 1: do defaulters have significantly lower income?

h0: mean income of defaulters >= mean income of non-defaulters  
h1: mean income of defaulters < mean income of non-defaulters  
test: independent samples t-test (one-tailed)

In [ ]:
defaulters = app[app['TARGET'] == 1]
non_defaulters = app[app['TARGET'] == 0]

income_defaulters = defaulters['AMT_INCOME_TOTAL'].dropna()
income_non_defaulters = non_defaulters['AMT_INCOME_TOTAL'].dropna()

t_stat, p_value_two = stats.ttest_ind(income_defaulters, income_non_defaulters, equal_var=False)
p_value_one = p_value_two / 2

np.mean(income_defaulters), np.mean(income_non_defaulters), t_stat, p_value_one

performs a one-tailed t-test to determine if defaulters have significantly lower income than non-defaulters

**Result:** REJECT the null hypothesis (p-value < 0.05). Defaulters have significantly lower income than non-defaulters.

#### hypothesis 2: is default rate different across genders?

h0: default rate is same for males and females  
h1: default rate differs between genders  
test: two-proportion z-test

In [ ]:
gender_data = app[app['CODE_GENDER'].isin(['M', 'F'])]
male_default = gender_data[gender_data['CODE_GENDER'] == 'M']['TARGET'].sum()
female_default = gender_data[gender_data['CODE_GENDER'] == 'F']['TARGET'].sum()
male_total = (gender_data['CODE_GENDER'] == 'M').sum()
female_total = (gender_data['CODE_GENDER'] == 'F').sum()

count_arr = np.array([male_default, female_default])
nobs_arr = np.array([male_total, female_total])

z_stat, p_value = proportions_ztest(count_arr, nobs_arr)
male_default / male_total, female_default / female_total, z_stat, p_value

conducts a two-proportion z-test to check if default rates differ significantly between males and females

**Result:** REJECT the null hypothesis (p-value < 0.05). Default rate is significantly different between genders.

#### hypothesis 3: is education level associated with default?

h0: education level and default are independent  
h1: education level and default are related  
test: chi-square test of independence

In [ ]:
contingency_table = pd.crosstab(app['NAME_EDUCATION_TYPE'], app['TARGET'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

chi2, p_value, dof

performs a chi-square test to determine if education level is associated with default status

**Result:** REJECT the null hypothesis (p-value < 0.05). Education level is significantly associated with default status.

#### hypothesis 4: do defaulters have higher credit-to-income ratio?

h0: mean credit-to-income ratio is same for both groups  
h1: defaulters have higher credit-to-income ratio  
test: independent samples t-test (one-tailed)

In [ ]:
ratio_defaulters = defaulters['CREDIT_INCOME_RATIO'].dropna()
ratio_non_defaulters = non_defaulters['CREDIT_INCOME_RATIO'].dropna()

t_stat, p_value_two = stats.ttest_ind(ratio_defaulters, ratio_non_defaulters, equal_var=False)
p_value_one = p_value_two / 2 if t_stat > 0 else 1 - p_value_two / 2

np.mean(ratio_defaulters), np.mean(ratio_non_defaulters), t_stat, p_value_one

tests if defaulters have a significantly higher credit-to-income ratio using a one-tailed t-test

**Result:** REJECT the null hypothesis (p-value < 0.05). Defaulters have significantly higher credit-to-income ratio.

#### hypothesis 5: does ext_source_2 differ between defaulters and non-defaulters?

h0: mean ext_source_2 is same for both groups  
h1: mean ext_source_2 differs between groups  
test: independent samples t-test

In [ ]:
ext2_defaulters = defaulters['EXT_SOURCE_2'].dropna()
ext2_non_defaulters = non_defaulters['EXT_SOURCE_2'].dropna()

t_stat, p_value = stats.ttest_ind(ext2_defaulters, ext2_non_defaulters, equal_var=False)

np.mean(ext2_defaulters), np.mean(ext2_non_defaulters), t_stat, p_value

performs a two-tailed t-test to check if external source 2 scores differ between defaulters and non-defaulters

**Result:** REJECT the null hypothesis (p-value < 0.05). EXT_SOURCE_2 is significantly different, with defaulters having lower scores.

#### hypothesis 6: is default rate different across income types?

h0: default rate is same across all income types  
h1: default rate differs across income types  
test: chi-square test of independence

In [ ]:
income_contingency = pd.crosstab(app['NAME_INCOME_TYPE'], app['TARGET'])
chi2, p_value, dof, expected = chi2_contingency(income_contingency)

chi2, p_value, dof

uses chi-square test to determine if default rates vary significantly across different income types

**Result:** REJECT the null hypothesis (p-value < 0.05). Default rate significantly varies across income types.

#### hypothesis 7: do defaulters have lower external source scores?

h0: mean ext_source_3 is same for both groups  
h1: defaulters have lower ext_source_3  
test: independent samples t-test (one-tailed)

In [ ]:
ext3_defaulters = defaulters['EXT_SOURCE_3'].dropna()
ext3_non_defaulters = non_defaulters['EXT_SOURCE_3'].dropna()

t_stat, p_value_two = stats.ttest_ind(ext3_defaulters, ext3_non_defaulters, equal_var=False)
p_value_one = p_value_two / 2

np.mean(ext3_defaulters), np.mean(ext3_non_defaulters), t_stat, p_value_one

performs a one-tailed t-test to verify if defaulters have significantly lower external source 3 scores

**Result:** REJECT the null hypothesis (p-value < 0.05). Defaulters have significantly lower EXT_SOURCE_3 scores.

#### hypothesis 8: variance in credit amount - defaulters vs non-defaulters

h0: variance in credit amount is same for both groups  
h1: variance differs between groups  
test: f-test for variance

In [ ]:
credit_defaulters = defaulters['AMT_CREDIT'].dropna()
credit_non_defaulters = non_defaulters['AMT_CREDIT'].dropna()

f_stat = np.var(credit_defaulters, ddof=1) / np.var(credit_non_defaulters, ddof=1)
df1 = len(credit_defaulters) - 1
df2 = len(credit_non_defaulters) - 1
p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), stats.f.sf(f_stat, df1, df2))

np.var(credit_defaulters, ddof=1), np.var(credit_non_defaulters, ddof=1), f_stat, p_value

conducts an f-test to check if the variance in credit amounts differs between defaulters and non-defaulters

**Result:** REJECT the null hypothesis (p-value < 0.05). Variance in credit amount differs significantly between defaulters and non-defaulters.